### Data Quality Issues Summary

In this notebook, the different data quality checks developed in the previous steps are combined into a single, simplified view.

The goal is not to introduce new calculations, but to bring together the main signals — missing values, validation results, and outliers — into a format that is easier to read and interpret from a business perspective.

This final table is designed to highlight where data quality issues are concentrated across assets.

In [ ]:
import pandas as pd 
import numpy as np
from src_functions import completeness
from src_functions import flag_validation
from src_functions import data_validation
from src_functions import outlier_and_returns

market_data = pd.read_csv(r"market_data.csv").drop(columns='Unnamed: 0')

In [2]:
completeness_table = completeness(market_data)
validation_df = flag_validation(market_data)
validation_heatmap = data_validation(validation_df)
outliers_table = outlier_and_returns(market_data)

In [3]:
from src_functions import summary_table

summary_missing_value = summary_table(market_data)

### Missing Values Summary

Missing values are computed at the asset level by counting the number of null observations across the main fields used in the analysis (close, volume, bid, ask, spread).

The focus here is on absolute counts rather than percentages, as they provide a clearer view of the magnitude of the issue.

All missing values are aggregated per asset to produce a single missing count, representing the overall data availability for each instrument.

### Validation Overview

Validation checks were applied to ensure that available data points satisfy basic consistency rules (for example, positive prices and valid bid/ask relationships).

In this dataset, all non-null observations passed the validation checks, resulting in no invalid records.

For this reason, the invalid count is set to zero for all assets in the final summary.

### Outlier Detection

Outliers are identified using log returns and a simple statistical threshold based on mean and standard deviation.

These observations represent unusually large price movements and are included as a separate data quality signal.

While not necessarily errors, they are relevant from a monitoring perspective, as they may indicate extreme market conditions or data anomalies.

In [4]:
final_summary = pd.concat([summary_missing_value, outliers_table.drop(columns=['neg_outliers_pct_on_tot', 'neg_outliers_pct_on_tot_outliers'])], axis=1)

# add invalid count (validation = 100% for all assets -> invalid = 0)
final_summary['invalid_count'] = 0

final_summary = final_summary.rename(
    columns={
        'tot_missing_value': 'missing_count',
        'tot_outliers_count': 'outlier_count'
    }
)

# reorder columns
final_summary = final_summary[['missing_count', 'invalid_count', 'outlier_count']]

final_summary

,missing_count,invalid_count,outlier_count
asset,,,
AAPL,195,0,13
EURUSD=X,5,0,11
GLD,195,0,13
SPY,195,0,12
TLT,195,0,7


### Final Issues Table

The final table combines the three main data quality components:

- Missing values (data availability)
- Invalid observations (rule-based checks)
- Outliers (extreme movements)

Each asset is represented with a count for each issue type.

This structure provides a compact and practical overview that can be easily used for reporting or further analysis.